In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244080204   # ← 改成你的学号
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp01
[学号] 20244080204
[机器指纹] 00582d46724e0b27
[RUN_ID] 20244080204-00582d46724e0b27
[Python] 3.10.22


实验1：正则表达式的基础匹配

In [2]:
import re

任务1：\d+、\s+、\bcat\b

In [3]:
s1 = "cat on mat; later: cats and dogs"
print(r"\d+  ->", re.findall(r"\d+", s1))     # 无数字
print(r"\s+  ->", re.findall(r"\s+", s1))     # 6 个空格
print(r"\bcat\b ->", re.findall(r"\bcat\b", s1))  # 只匹配独立 cat，cats 不匹配

\d+  -> []
\s+  -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b -> ['cat']


Markdown 说明要点：

\d+：一个或多个数字，句中无数字，返回 []。

\s+：一个或多个空白，返回 [' ', ...] 共 6 个。

\bcat\b：\b 为单词边界，cats 中 cat 后接 s，t 后无边界，故不匹配。

任务2：交替与顺序差异

In [4]:
s2 = "cat dog bird CAT"
print(r"cat|dog ->", re.findall(r"cat|dog", s2))

print(r"a|aa|aaa on 'aaaa' ->", re.findall(r"a|aa|aaa", "aaaa"))
print(r"aaa|aa|a on 'aaaa' ->", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog -> ['cat', 'dog']
a|aa|aaa on 'aaaa' -> ['a', 'a', 'a', 'a']
aaa|aa|a on 'aaaa' -> ['aaa', 'a']


说明：交替 | 按从左到右优先匹配。

a|aa|aaa：每个位置先试 a，故得 ['a','a','a','a']。

aaa|aa|a：位置 0 先试 aaa，剩 a，得 ['aaa','a']。

大写 CAT 因为默认区分大小写，不会匹配 cat。

任务3：量词对比

In [5]:
s3 = "ct cat caat caaat"
for pat in [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]:
    print(pat, "->", re.findall(pat, s3))

ca*t -> ['ct', 'cat', 'caat', 'caaat']
ca+t -> ['cat', 'caat', 'caaat']
ca?t -> ['ct', 'cat']
ca{2}t -> ['caat']
ca{1,2}t -> ['cat', 'caat']


实验2：分组、回溯引用与替换
任务1：提取邮箱用户名与域名

In [6]:
s = "联系: alice@example.com; 备选: bob@test.org"
res = re.findall(r"([\w.-]+)@([\w.-]+)", s)
for user, domain in res:
    print(f"用户名: {user}, 域名: {domain}")

用户名: alice, 域名: example.com
用户名: bob, 域名: test.org


说明：两个捕获组分别对应 @ 前后的用户名与域名；[\w.-]+ 允许字母、数字、下划线、点、短横。

任务2：后向引用找连续重复单词

In [7]:
s = "cat cat, dog cat, bird bird"
print(re.findall(r"\b(\w+)\s+\1\b", s))   # ['cat', 'bird']

['cat', 'bird']


说明：(\w+) 捕获单词，\1 引用同一内容；dog cat 因两词不同不匹配。

任务3：日期 年-月-日 → 月-日-年

In [8]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前:\n" + dates)
new_dates = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", dates)
print("转换后:\n" + new_dates)

转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05-02-2024
05-03-2024


说明：[-/.] 统一分隔符；替换中 \2-\3-\1 重排三组；re.sub 默认全局替换且不改变换行，行数不变。

任务4：电话号码统一为 (123) 456-7890

In [9]:
phones = "联系我: 123-456-7890 或 987.654.3210"
new_phones = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", phones)
print(new_phones)

联系我: (123) 456-7890 或 (987) 654-3210


实验3：用 BeautifulSoup 解析 HTML
构建解析树 + 任务1

In [10]:
from bs4 import BeautifulSoup

html_doc = """<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲: 正则表达式</p>
<p class="course">第七讲: 结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>"""

soup = BeautifulSoup(html_doc, "html.parser")

print("title 文本:", soup.title.get_text())
h1 = soup.h1
print("h1 标签名:", h1.name)
print("h1 文本:", h1.get_text())

title 文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


任务2：find_all 取段落和链接

In [11]:
courses = soup.find_all("p", class_="course")
print("class=course 段落数:", len(courses))
for i, p in enumerate(courses, 1):
    print(f"  段落{i}: {p.get_text()}")

links = soup.find_all("a")
print("链接数:", len(links))
for a in links:
    print(f"  文字: {a.get_text()}  |  href: {a['href']}")

class=course 段落数: 2
  段落1: 第六讲: 正则表达式
  段落2: 第七讲: 结构化数据
链接数: 2
  文字: 讲义六  |  href: https://example.org/lecture06
  文字: 讲义七  |  href: https://example.org/lecture07


任务3：get_text() 提取可见文本并对比

In [12]:
print("=== 整个文档可见文本 ===")
print(repr(soup.get_text()))

print("=== 去除多余空白后 ===")
print(soup.get_text(strip=True))

print("=== 单个段落 ===")
print(repr(courses[0].get_text()))

print("=== 单个链接 ===")
print(repr(links[0].get_text()))

=== 整个文档可见文本 ===
'实验课程\n数据分析\n第六讲: 正则表达式\n第七讲: 结构化数据\n讲义六\n讲义七\n'
=== 去除多余空白后 ===
实验课程数据分析第六讲: 正则表达式第七讲: 结构化数据讲义六讲义七
=== 单个段落 ===
'第六讲: 正则表达式'
=== 单个链接 ===
'讲义六'


**实验3 任务3：get_text() 解析结果对比说明**
1. `soup.get_text()`：提取了整个 HTML 文档（包括 `<title>`）的所有可见文本，并且保留了原始 HTML 源码中的换行符（`\n`）。
2. `soup.get_text(strip=True)`：在提取文本的同时去除了每个文本块前后的空白字符和换行符，将全篇文本紧凑地连接在了一起，适合做纯文本的快速预览。
3. `courses[0].get_text()` 和 `links[0].get_text()`：只提取了特定标签内部的文本（即单个段落的内容或单个链接的文字），不包含周围的 HTML 结构，也不包含标签的属性（如链接的 href）。